# Country samples for preliminary EIS analysis

Run the cells in order. Edit **Country selection and sampling controls**, review
the comparison table, then run **Export reviewed samples**. No OLS is fitted here.

The World Bank inputs are final consumption expenditure (constant 2015 US$,
`NE.CON.TOTL.KD`) and real interest rate (%, `FR.INR.RINR`). The module preserves
consumption levels and converts interest rates to decimals (5% becomes 0.05).

## Regression Model
Consider the simple regression model to estimate Elasticity of Intertemporal Substitution (EIS) by assumming $EIS = 1/\gamma = \alpha_1$ $$ \frac{c_{t+1} -c_{t}}{c_{t}}  = \alpha_0 + \alpha_1 r_{t+1} + \epsilon_{t+1}$$ where $c$ is final consumption, $r$ is real interest rate.


In [1]:
import numpy as np
import pandas as pd
import statsmodels as sm
from pathlib import Path
from IPython.display import display

In [2]:
# Works when the kernel starts in the repository root or notebook/.
working_directory = Path.cwd().resolve()
project_root = next(
    (directory for directory in (working_directory, *working_directory.parents)
     if (directory / "data/world_dev_index/cdata.xls").is_file()
     and (directory / "data/world_dev_index/rdata.xls").is_file()),
    None,
)
if project_root is None:
    raise FileNotFoundError(
        "Start the kernel in the econ-consumption repository or notebook directory; "
        "expected data/world_dev_index/cdata.xls and rdata.xls."
    )

data_directory = project_root / "data" / "world_dev_index"
df_cdata = pd.read_excel(data_directory / "cdata.xls", engine="xlrd", skiprows=3)
df_rdata = pd.read_excel(data_directory / "rdata.xls", engine="xlrd", skiprows=3)

In [3]:
df_rdata

,Country Name,Country Code,Indicator Name,Indicator Code,1960,1961,1962,1963,1964,1965,...,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
0,Aruba,ABW,Real interest rate (%),FR.INR.RINR,NaN,NaN,NaN,NaN,NaN,NaN,...,7.467580,6.143280,3.268734,4.717854,10.492888,4.073238,1.926068,2.882217,NaN,NaN
1,Africa Eastern and Southern,AFE,Real interest rate (%),FR.INR.RINR,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Afghanistan,AFG,Real interest rate (%),FR.INR.RINR,NaN,NaN,NaN,NaN,NaN,NaN,...,17.583938,12.141178,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Africa Western and Central,AFW,Real interest rate (%),FR.INR.RINR,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Angola,AGO,Real interest rate (%),FR.INR.RINR,NaN,NaN,NaN,NaN,NaN,NaN,...,-4.209573,-6.401132,-3.165277,-0.793755,2.947088,-11.944247,1.735822,-2.022156,-6.628120,-0.931745
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
260,Kosovo,XKX,Real interest rate (%),FR.INR.RINR,NaN,NaN,NaN,NaN,NaN,NaN,...,6.615990,6.364675,5.079009,NaN,NaN,NaN,NaN,NaN,NaN,NaN
261,"Yemen, Rep.",YEM,Real interest rate (%),FR.INR.RINR,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
262,South Africa,ZAF,Real interest rate (%),FR.INR.RINR,NaN,5.200152,6.013274,2.423273,3.667879,3.800071,...,3.278252,4.647315,5.856184,5.268416,2.198414,0.595088,3.373876,6.472067,7.417540,7.683541
263,Zambia,ZMB,Real interest rate (%),FR.INR.RINR,NaN,NaN,NaN,NaN,NaN,NaN,...,1.693929,2.091490,2.215865,2.469674,-3.893032,NaN,NaN,NaN,NaN,NaN


## Import the sampling module

The reusable implementation lives in `notebook/src/eis_sampling/sampling.py`.
See [the module README](src/eis_sampling/README.md) for Python and terminal examples.

`prepare_country_samples` returns `(coverage, samples)`. `samples` is keyed by
World Bank country code; every DataFrame has exactly the three export columns.
Source DataFrames are left unchanged.

Coverage and internal gaps describe the full source history. Complete overlap
intervals and selected periods respect your optional year bounds. An internal
gap means a missing year between a series' first and last available years.

- `per_country`: longest consecutive complete overlap for each country.
- `shared`: longest consecutive complete overlap across **all** selected countries.
- Equal-length runs: choose the most recent. No interpolation or filling.
- Unavailable samples remain visible in the report and are skipped at export.
  If there is no shared window, every shared sample is empty.

In [4]:
from pathlib import Path
import sys

# Locate the repository even when the kernel starts in notebook/.
cwd = Path.cwd().resolve()
project_root = next(
    (directory for directory in (cwd, *cwd.parents)
     if (directory / "data/world_dev_index").is_dir()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Start Python in the econ-consumption repository or a subfolder.")

# Support either source layout; the package currently lives in notebook/src/.
source_directory = next(
    (directory for directory in (project_root / "src", project_root / "notebook/src")
     if (directory / "eis_sampling/__init__.py").is_file()),
    None,
)
if source_directory is None:
    raise ModuleNotFoundError(f"Cannot find eis_sampling under {project_root}.")
if str(source_directory) not in sys.path:
    sys.path.insert(0, str(source_directory))

from eis_sampling import prepare_country_samples, export_country_samples


## Country selection and sampling controls

Use a code or exact World Bank name, ignoring case: `"VNM"` or `"Viet Nam"`.
For a comparison, try `countries = ["VNM", "USA", "JPN"]`. Use `"shared"` for
identical export years across countries. Leave year bounds as `None` to use all years.

To look up names, inspect
`df_cdata[["Country Name", "Country Code"]].sort_values("Country Name")`.
World Bank regional aggregates remain selectable if explicitly requested.

In [5]:
countries = ["VNM"]
sample_mode = "per_country"  # "per_country" or "shared"
start_year = None             # inclusive, e.g. 2000
end_year = None               # inclusive, e.g. 2020
output_directory = data_directory / "samples"

In [6]:
coverage, samples = prepare_country_samples(
    df_cdata, df_rdata, countries,
    sample_mode=sample_mode, start_year=start_year, end_year=end_year,
)
# Countries appear side by side; the original coverage table remains available.
with pd.option_context("display.max_colwidth", None, "display.max_columns", None,
                       "display.max_rows", None):
    display(coverage.set_index(["country_code", "country_name"]).T)
for code, sample in samples.items():
    print(f"{sample.attrs['country_name']} ({code}): {len(sample)} sample rows")
    display(sample.head())

country_code,VNM
country_name,Viet Nam
consumption_start,1994
consumption_end,2025
consumption_observations,32
consumption_missing_years,[]
interest_rate_start,1993
interest_rate_end,2023
interest_rate_observations,29
interest_rate_missing_years,"[1994, 1995]"
overlap_intervals,[1996–2023]


Viet Nam (VNM): 28 sample rows


,year,final_consumption,real_interest_rate
0,1996,5.436740e+10,0.104909
1,1997,5.749536e+10,0.073353
2,1998,6.000444e+10,0.051105
3,1999,6.116162e+10,0.065875
4,2000,6.313117e+10,0.069058


## Export reviewed samples

The next cell writes the samples shown above. After changing the controls, rerun
the preparation cell before exporting. Rerunning export replaces files for the
selected nonempty samples; unrelated files are left in place. A skipped country
does not delete a previous export: check the returned manifest for current results.

Each CSV contains `year`, `final_consumption`, `real_interest_rate`, with no index.
Viet Nam's filename is `viet_nam_data.csv`.

**For your later OLS:** a row labeled year \(y\) contains \(c_y\) and \(r_y\).
Compute \(g_y = (c_y-c_{y-1})/c_{y-1}\), then regress \(g_y\) on an intercept
and the **same row's** \(r_y\). Equivalently, \(y=t+1\) in your equation.
Rates are already decimal, so do **not** divide them by 100 again.

In Python, after sorting by year, growth can be computed with
`sample["final_consumption"].pct_change(fill_method=None)`. Drop its first missing
value before OLS. With complete, consecutive levels, \(N\) exported rows supply at
most \(N-1\) growth observations. No prior-year baseline row is added. Samples with
fewer than three level rows are flagged; three rows give only two growth observations
and no residual degrees of freedom for an intercept-and-slope regression.

With the supplied data, Viet Nam selects **1996–2023** (28 level rows, 27 growth
observations). Its interest-rate series has internal gaps in **1994–1995**.
Selecting VNM, USA and JPN in shared mode gives **1996–2017**.

In [7]:
export_manifest = export_country_samples(samples, output_directory)
display(export_manifest)

,country_code,country_name,path,rows,status
0,VNM,Viet Nam,/Users/nguyenthang/Repositories/econ-consumpti...,28,Exported
